### Step 1 — Project setup and imports

In [9]:
# Core imports
import numpy as np
from pathlib import Path
import sys

# Base paths
PROJECT_ROOT = Path("..")
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Allow imports from ../src
sys.path.append(str(PROJECT_ROOT / "src"))

# Import our utilities
from bbo_utils import (
    load_xy,
    fit_gp,
    acquisition_ucb,
    generate_random_candidates,
    format_submission,
    validate_submission_string,
)

#from candidates import generate_random_candidates

### Step 2 — Load and validate initial data

In [10]:
for func_id in range(1, 9):
    X, y = load_xy(func_id, DATA_RAW_DIR)

    print(
        f"Function {func_id}: "
        f"X shape={X.shape}, "
        f"y shape={y.shape}, "
        f"X range=({X.min():.6f}, {X.max():.6f}), "
        f"y range=({y.min():.6f}, {y.max():.6f})"
    )



Function 1: X shape=(10, 2), y shape=(10,), X range=(0.078723, 0.883890), y range=(-0.003606, 0.000000)
Function 2: X shape=(10, 2), y shape=(10,), X range=(0.028698, 0.926564), y range=(-0.065624, 0.611205)
Function 3: X shape=(15, 3), y shape=(15,), X range=(0.046809, 0.990882), y range=(-0.398926, -0.034835)
Function 4: X shape=(30, 4), y shape=(30,), X range=(0.006250, 0.999483), y range=(-32.625660, -4.025542)
Function 5: X shape=(20, 4), y shape=(20,), X range=(0.038193, 0.957644), y range=(0.112940, 1088.859618)
Function 6: X shape=(20, 5), y shape=(20,), X range=(0.004911, 0.978806), y range=(-2.571170, -0.714265)
Function 7: X shape=(30, 6), y shape=(30,), X range=(0.003635, 0.998655), y range=(0.002701, 1.364968)
Function 8: X shape=(40, 8), y shape=(40,), X range=(0.003419, 0.998885), y range=(5.592193, 9.598482)


## Section 2 — Single-function Bayesian Optimisation Walkthrough

### 2.1 Choose the function to explore

In [3]:
FUNC_ID = 1  # change to 2..8 when you want to inspect another function

### 2.2 Load data for the chosen function

In [5]:
X, y = load_xy(FUNC_ID, DATA_RAW_DIR)

dim = X.shape[1]

print("Function:", FUNC_ID)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("dim:", dim)
print("Best y so far:", float(y.max()))

# print("X:")
# print(X)
# print("y:")
# print(y)

Function: 1
X shape: (10, 2)
y shape: (10,)
dim: 2
Best y so far: 7.710875114502849e-16


### 2.3 Fit the Gaussian Process surrogate model

In [6]:
gp = fit_gp(X, y)
print("GP fitted for Function", FUNC_ID)

GP fitted for Function 1


### 2.4 Generate candidate points in the valid domain

In [7]:
N_CANDIDATES = 5000  # increase later if you want

X_candidates = generate_random_candidates(
    n_candidates=N_CANDIDATES,
    dim=dim,
)

print("Candidates shape:", X_candidates.shape)
print("Candidates range:", float(X_candidates.min()), float(X_candidates.max()))


Candidates shape: (5000, 2)
Candidates range: 0.00031441964408696066 0.9999770204176942


### 2.5 Predict mean and uncertainty for all candidates

In [8]:
mean, std = gp.predict(X_candidates, return_std=True)

print("mean range:", float(mean.min()), float(mean.max()))
print("std range:", float(std.min()), float(std.max()))

mean range: -0.0036053148149808437 -1.6458692254929985e-05
std range: 2.3222454250804766e-05 0.001081819512495705


### 2.6 Compute acquisition scores (UCB)

In [9]:
KAPPA = 2.0  # exploration vs exploitation control

scores = acquisition_ucb(mean, std, kappa=KAPPA)

print("scores range:", float(scores.min()), float(scores.max()))

scores range: -0.0035588699064792342 0.0018328774103059872


### 2.7 Select the best next point

In [10]:
best_idx = int(scores.argmax())
x_next = X_candidates[best_idx]

print("x_next:", x_next)
print("x_next dim:", len(x_next))

x_next: [0.70573805 0.87198986]
x_next dim: 2


### 2.8 Format the submission string (portal format)

In [11]:
submission = format_submission(x_next)
ok, msg = validate_submission_string(submission, dim=dim)

print("submission:", submission)
print("valid:", ok, "|", msg)

submission: 0.705738-0.871990
valid: True | OK


### 2.9 (Optional) Compare against best observed y

In [12]:
print("Best observed y so far:", float(y.max()))
print("Note: x_next is NOT evaluated yet — it will be evaluated after portal submission.")

Best observed y so far: 7.710875114502849e-16
Note: x_next is NOT evaluated yet — it will be evaluated after portal submission.


## Section 3 — Week 1 (Module 12) : Generate Final Submissions for All 8 Functions

In [3]:
N_CANDIDATES = 5000
KAPPA = 2.0

results = {}

for func_id in range(1, 9):
    # Load data
    X, y = load_xy(func_id, DATA_RAW_DIR)
    dim = X.shape[1]

    # Fit GP surrogate
    gp = fit_gp(X, y)

    # Candidate pool
    X_candidates = generate_random_candidates(
        n_candidates=N_CANDIDATES,
        dim=dim,
    )

    # Predict mean + uncertainty
    mean, std = gp.predict(X_candidates, return_std=True)

    # Acquisition (UCB)
    scores = acquisition_ucb(mean, std, kappa=KAPPA)

    # Pick best candidate
    x_next = X_candidates[int(scores.argmax())]

    # Format and validate portal submission
    submission = format_submission(x_next)
    ok, msg = validate_submission_string(submission, dim=dim)

    if not ok:
        raise ValueError(f"Function {func_id} invalid submission: {msg} -> {submission}")

    results[func_id] = submission
    print(f"Function {func_id}: {submission}")


Function 1: 0.705738-0.871990
Function 2: 0.828497-0.970137
Function 3: 0.418575-0.366581-0.521443
Function 4: 0.370752-0.426131-0.337047-0.465539
Function 5: 0.398348-0.896184-0.926939-0.972836
Function 6: 0.296485-0.322087-0.332446-0.760368-0.040944
Function 7: 0.033169-0.329738-0.366358-0.234301-0.286315-0.704927
Function 8: 0.147947-0.193825-0.060981-0.008004-0.719306-0.453461-0.045592-0.938970


## [Week 2] - Section 4 (Module 13 -Logistic Regression): Add Week 1 results and generate Week 2 queries

### 4.1 Week 1 submitted inputs and observed outputs

In [15]:
# Week 1 submitted inputs (portal)
WEEK1_INPUTS = {
    1: [0.705738, 0.871990],
    2: [0.828497, 0.970137],
    3: [0.418575, 0.366581, 0.521443],
    4: [0.370752, 0.426131, 0.337047, 0.465539],
    5: [0.398348, 0.896184, 0.926939, 0.972836],
    6: [0.296485, 0.322087, 0.332446, 0.760368, 0.040944],
    7: [0.033169, 0.329738, 0.366358, 0.234301, 0.286315, 0.704927],
    8: [0.147947, 0.193825, 0.060981, 0.008004, 0.719306, 0.453461, 0.045592, 0.938970],
}

# Week 1 observed outputs (email)
WEEK1_OUTPUTS = {
    1: 8.994904739436242e-46,
    2: 0.07492805470733854,
    3: -0.013221116845993724,
    4: -0.4713345507601363,
    5: 2360.0402820167205,
    6: -0.49886770748675036,
    7: 2.4608579737515917,
    8: 9.894432442301,
}


### 4.2 Helper: load initial data and add Week 1 point (returns updated dataset)

In [11]:
import numpy as np

def load_xy_with_week1(func_id: int, data_raw_dir):
    """
    Load initial (X, y) and add Week 1 (x, y) to build the Week 2 training dataset.

    Returns:
      X_updated: shape (11, dim)
      y_updated: shape (11,)
    """
    # 10 initial points
    X, y = load_xy(func_id, data_raw_dir)

    # 1 point from Week 1
    x_new = np.array(WEEK1_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_new = np.array([WEEK1_OUTPUTS[func_id]], dtype=float)

    # Dim check (safety)
    if x_new.shape[1] != X.shape[1]:
        raise ValueError(
            f"Function {func_id}: dimension mismatch. "
            f"initial dim={X.shape[1]}, week1 dim={x_new.shape[1]}"
        )

    # Updated dataset (11 points)
    X_updated = np.vstack([X, x_new])
    y_updated = np.concatenate([y, y_new])

    return X_updated, y_updated


### 4.3 Sanity check: confirm all functions now have 11 points

In [6]:
for func_id in range(1, 9):
    X_updated, y_updated = load_xy_with_week1(func_id, DATA_RAW_DIR)
    print(f"Function {func_id}: X={X_updated.shape}, y={y_updated.shape}, dim={X_updated.shape[1]}")

Function 1: X=(11, 2), y=(11,), dim=2
Function 2: X=(11, 2), y=(11,), dim=2
Function 3: X=(16, 3), y=(16,), dim=3
Function 4: X=(31, 4), y=(31,), dim=4
Function 5: X=(21, 4), y=(21,), dim=4
Function 6: X=(21, 5), y=(21,), dim=5
Function 7: X=(31, 6), y=(31,), dim=6
Function 8: X=(41, 8), y=(41,), dim=8


### 4.4 Generate Week 2 submission strings for all 8 functions

In [7]:
N_CANDIDATES = 5000
KAPPA = 2.0

results_week2 = {}

for func_id in range(1, 9):
    # Updated training data: 10 initial + Week 1 = 11 points
    X_updated, y_updated = load_xy_with_week1(func_id, DATA_RAW_DIR)
    dim = X_updated.shape[1]

    # Fit GP on updated data
    gp = fit_gp(X_updated, y_updated)

    # Candidates in [0,1)^dim
    X_candidates = generate_random_candidates(
        n_candidates=N_CANDIDATES,
        dim=dim,
    )

    # Predict + score
    mean, std = gp.predict(X_candidates, return_std=True)
    scores = acquisition_ucb(mean, std, kappa=KAPPA)

    # Select best candidate and format
    x_next = X_candidates[int(scores.argmax())]
    submission = format_submission(x_next)

    # Validate portal format
    ok, msg = validate_submission_string(submission, dim=dim)
    if not ok:
        raise ValueError(f"Function {func_id}: invalid submission: {msg} -> {submission}")

    results_week2[func_id] = submission
    print(f"Week 2 - Function {func_id}: {submission}")



Week 2 - Function 1: 0.763923-0.799245
Week 2 - Function 2: 0.692932-0.920964
Week 2 - Function 3: 0.388137-0.056619-0.408877
Week 2 - Function 4: 0.413742-0.470182-0.290309-0.439853
Week 2 - Function 5: 0.901642-0.984569-0.982367-0.952999
Week 2 - Function 6: 0.347816-0.270027-0.588964-0.987295-0.290167
Week 2 - Function 7: 0.196139-0.369342-0.400707-0.230683-0.233984-0.607869
Week 2 - Function 8: 0.078187-0.158932-0.026599-0.138226-0.529417-0.076738-0.059868-0.118162


### 4.5 Save Week 2 submissions

In [7]:
from pathlib import Path

out_dir = PROJECT_ROOT / "outputs" / "week_02"
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "submissions.txt"
with out_file.open("w", encoding="utf-8") as f:
    for func_id in range(1, 9):
        f.write(f"Function {func_id}: {results_week2[func_id]}\n")

print("Saved Week 2 submissions to:", out_file)


Saved Week 2 submissions to: ..\outputs\week_02\submissions.txt


### 4.6 Quick summary for reflection

In [8]:
for func_id in range(1, 9):
    X_updated, y_updated = load_xy_with_week1(func_id, DATA_RAW_DIR)
    print(
        f"Function {func_id}: dim={X_updated.shape[1]}, points={X_updated.shape[0]}, best_y_so_far={float(y_updated.max()):.6f}, {float(y_updated.max())}"
    )

Function 1: dim=2, points=11, best_y_so_far=0.000000, 7.710875114502849e-16
Function 2: dim=2, points=11, best_y_so_far=0.611205, 0.6112052157614438
Function 3: dim=3, points=16, best_y_so_far=-0.013221, -0.013221116845993724
Function 4: dim=4, points=31, best_y_so_far=-0.471335, -0.4713345507601363
Function 5: dim=4, points=21, best_y_so_far=2360.040282, 2360.0402820167205
Function 6: dim=5, points=21, best_y_so_far=-0.498868, -0.49886770748675036
Function 7: dim=6, points=31, best_y_so_far=2.460858, 2.4608579737515917
Function 8: dim=8, points=41, best_y_so_far=9.894432, 9.894432442301


In [ ]:
def best_result_so_far(func_id, data_raw_dir):
    # Load initial data to get the correct initial count for this function
    X0, y0 = load_xy(func_id, data_raw_dir)
    n0 = len(y0)  # initial number of points (varies by function)

    # Load updated data (initial + Week 1)
    X_updated, y_updated = load_xy_with_week1(func_id, data_raw_dir)

    best_idx = int(np.argmax(y_updated))
    best_value = float(y_updated[best_idx])

    # Determine source using dynamic n0 (NOT hardcoded 10)
    if best_idx < n0:
        source = "Initial data (provided)"
    elif best_idx == n0:
        source = "Week 1 submission"
    else:
        week_num = (best_idx - n0) + 1
        source = f"Week {week_num} submission"

    return {
        "function": func_id,
        "best_index": best_idx,
        "best_value": best_value,
        "source": source,
        "initial_points": n0,
        "total_points_so_far": len(y_updated),
    }

In [10]:
for func_id in range(1, 9):
    result = best_result_so_far(func_id, DATA_RAW_DIR)
    print(
        f"Function {result['function']}: "
        f"best y = {result['best_value']:.6f} | "
        f"index = {result['best_index']} | "
        f"source = {result['source']}"
    )

Function 1: best y = 0.000000 | index = 2 | source = Initial data (provided)
Function 2: best y = 0.611205 | index = 9 | source = Initial data (provided)
Function 3: best y = -0.013221 | index = 15 | source = Week 1 submission
Function 4: best y = -0.471335 | index = 30 | source = Week 1 submission
Function 5: best y = 2360.040282 | index = 20 | source = Week 1 submission
Function 6: best y = -0.498868 | index = 20 | source = Week 1 submission
Function 7: best y = 2.460858 | index = 30 | source = Week 1 submission
Function 8: best y = 9.894432 | index = 40 | source = Week 1 submission


## [Week 3] - Section 5 (Module 14 - SVM): Add Week 2 results and generate Week 3 queries

### 5.1 Store Week 2 inputs and outputs

In [16]:
# Week 2 submitted inputs (portal)
WEEK2_INPUTS = {
    1: [0.763923, 0.799245],
    2: [0.692932, 0.920964],
    3: [0.388137, 0.056619, 0.408877],
    4: [0.413742, 0.470182, 0.290309, 0.439853],
    5: [0.901642, 0.984569, 0.982367, 0.952999],
    6: [0.347816, 0.270027, 0.588964, 0.987295, 0.290167],
    7: [0.196139, 0.369342, 0.400707, 0.230683, 0.233984, 0.607869],
    8: [0.078187, 0.158932, 0.026599, 0.138226, 0.529417, 0.076738, 0.059868, 0.118162],
}

# Week 2 observed outputs (email)
WEEK2_OUTPUTS = {
    1: 1.1451214454803375e-33,
    2: 0.5227732568346157,
    3: -0.08200213652671233,
    4: -1.7902558622959037,
    5: 5823.4794607079075,
    6: -0.6884064929199594,
    7: 2.425181219664537,
    8: 9.6885056366981,
}


### 5.2 Helper: build Week 3 training dataset (initial + Week 1 + Week 2)

In [16]:
def load_xy_with_week1_and_week2(func_id: int, data_raw_dir):
    """
    Load initial (X, y) and add Week 1 + Week 2 points.
    Returns:
      X_updated: shape (n0 + 2, dim)
      y_updated: shape (n0 + 2,)
    """
    X0, y0 = load_xy(func_id, data_raw_dir)
    dim = X0.shape[1]

    x_w1 = np.array(WEEK1_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_w1 = np.array([WEEK1_OUTPUTS[func_id]], dtype=float)

    x_w2 = np.array(WEEK2_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_w2 = np.array([WEEK2_OUTPUTS[func_id]], dtype=float)

    if x_w1.shape[1] != dim or x_w2.shape[1] != dim:
        raise ValueError(f"Function {func_id}: dimension mismatch.")

    X_updated = np.vstack([X0, x_w1, x_w2])
    y_updated = np.concatenate([y0, y_w1, y_w2])

    return X_updated, y_updated


### 5.3 Sanity check: confirm dataset grew by 2 for all functions

In [17]:
for func_id in range(1, 9):
    X0, y0 = load_xy(func_id, DATA_RAW_DIR)
    X_updated, y_updated = load_xy_with_week1_and_week2(func_id, DATA_RAW_DIR)

    print(
        f"Function {func_id}: "
        f"initial_points={len(y0)} -> now_points={len(y_updated)} | "
        f"dim={X_updated.shape[1]}"
    )

Function 1: initial_points=10 -> now_points=12 | dim=2
Function 2: initial_points=10 -> now_points=12 | dim=2
Function 3: initial_points=15 -> now_points=17 | dim=3
Function 4: initial_points=30 -> now_points=32 | dim=4
Function 5: initial_points=20 -> now_points=22 | dim=4
Function 6: initial_points=20 -> now_points=22 | dim=5
Function 7: initial_points=30 -> now_points=32 | dim=6
Function 8: initial_points=40 -> now_points=42 | dim=8


### 5.4 Best-so-far tracker

In [22]:
def best_so_far_with_source(func_id: int, data_raw_dir):
    """
    Returns best y observed so far and where it came from:
      - Initial data
      - Week 1
      - Week 2

    Works even when initial dataset size differs by function.
    """
    # Initial data
    X0, y0 = load_xy(func_id, data_raw_dir)
    n0 = len(y0)  # initial point count for this function

    # Current dataset: initial + week1 + week2
    X_updated, y_updated = load_xy_with_week1_and_week2(func_id, data_raw_dir)

    best_idx = int(np.argmax(y_updated))
    best_y = float(y_updated[best_idx])
    best_x = X_updated[best_idx]

    # Determine source
    if best_idx < n0:
        source = "Initial data (provided)"
        week = 0
    elif best_idx == n0:
        source = "Week 1 submission"
        week = 1
    elif best_idx == n0 + 1:
        source = "Week 2 submission"
        week = 2
    else:
        # For future weeks (if you extend)
        week = (best_idx - n0) + 1
        source = f"Week {week} submission"

    return {
        "function": func_id,
        "dim": X0.shape[1],
        "initial_points": n0,
        "total_points": len(y_updated),
        "best_index": best_idx,
        "best_week": week,
        "best_y": best_y,
        "best_x": best_x,
        "source": source,
        "best_x_portal": format_submission(best_x),
    }


In [23]:
for func_id in range(1, 9):
    r = best_so_far_with_source(func_id, DATA_RAW_DIR)
    print(
        f"Function {r['function']} (dim={r['dim']}): "
        f"best_y={r['best_y']:.6f} | source={r['source']} | "
        f"index={r['best_index']} | x={r['best_x_portal']}"
    )

Function 1 (dim=2): best_y=0.000000 | source=Initial data (provided) | index=2 | x=0.731024-0.733000
Function 2 (dim=2): best_y=0.611205 | source=Initial data (provided) | index=9 | x=0.702637-0.926564
Function 3 (dim=3): best_y=-0.013221 | source=Week 1 submission | index=15 | x=0.418575-0.366581-0.521443
Function 4 (dim=4): best_y=-0.471335 | source=Week 1 submission | index=30 | x=0.370752-0.426131-0.337047-0.465539
Function 5 (dim=4): best_y=5823.479461 | source=Week 2 submission | index=21 | x=0.901642-0.984569-0.982367-0.952999
Function 6 (dim=5): best_y=-0.498868 | source=Week 1 submission | index=20 | x=0.296485-0.322087-0.332446-0.760368-0.040944
Function 7 (dim=6): best_y=2.460858 | source=Week 1 submission | index=30 | x=0.033169-0.329738-0.366358-0.234301-0.286315-0.704927
Function 8 (dim=8): best_y=9.894432 | source=Week 1 submission | index=40 | x=0.147947-0.193825-0.060981-0.008004-0.719306-0.453461-0.045592-0.938970


In [24]:
def week_progress_check(func_id: int, data_raw_dir):
    X0, y0 = load_xy(func_id, data_raw_dir)
    n0 = len(y0)

    # Build full dataset
    X_updated, y_updated = load_xy_with_week1_and_week2(func_id, data_raw_dir)

    best_initial = float(np.max(y_updated[:n0]))
    y_week1 = float(y_updated[n0])
    y_week2 = float(y_updated[n0 + 1])
    best_overall = float(np.max(y_updated))

    return {
        "function": func_id,
        "best_initial": best_initial,
        "week1_y": y_week1,
        "week2_y": y_week2,
        "best_overall": best_overall,
    }

for func_id in range(1, 9):
    r = week_progress_check(func_id, DATA_RAW_DIR)
    print(
        f"Function {func_id}: "
        f"best_initial={r['best_initial']:.6f} | "
        f"week1={r['week1_y']:.6f} | week2={r['week2_y']:.6f} | "
        f"best_overall={r['best_overall']:.6f}"
    )


Function 1: best_initial=0.000000 | week1=0.000000 | week2=0.000000 | best_overall=0.000000
Function 2: best_initial=0.611205 | week1=0.074928 | week2=0.522773 | best_overall=0.611205
Function 3: best_initial=-0.034835 | week1=-0.013221 | week2=-0.082002 | best_overall=-0.013221
Function 4: best_initial=-4.025542 | week1=-0.471335 | week2=-1.790256 | best_overall=-0.471335
Function 5: best_initial=1088.859618 | week1=2360.040282 | week2=5823.479461 | best_overall=5823.479461
Function 6: best_initial=-0.714265 | week1=-0.498868 | week2=-0.688406 | best_overall=-0.498868
Function 7: best_initial=1.364968 | week1=2.460858 | week2=2.425181 | best_overall=2.460858
Function 8: best_initial=9.598482 | week1=9.894432 | week2=9.688506 | best_overall=9.894432


### 5.5 Fit GP using standardised y

In [ ]:
def fit_gp_with_y_standardization(X, y, *, length_scale=0.2, kernel_amplitude=1.0, random_state=42):
    """
    Standardize y -> fit GP -> return (gp, y_mean, y_std).
    We keep gp trained on standardized y.
    """
    y = np.asarray(y).reshape(-1)
    y_mean = float(np.mean(y))
    y_std = float(np.std(y)) if float(np.std(y)) > 0 else 1.0

    y_stdzd = (y - y_mean) / y_std
    gp = fit_gp(X, y_stdzd, length_scale=length_scale, kernel_amplitude=kernel_amplitude, random_state=random_state)
    return gp, y_mean, y_std

### 5.6 Generate Week 3 submissions for all 8 functions (with dimension-aware kappa)

In [26]:
N_CANDIDATES = 10000  # small upgrade (more candidates helps in higher dimensions)
BASE_KAPPA = 2.0

results_week3 = {}

for func_id in range(1, 9):
    X_updated, y_updated = load_xy_with_week1_and_week2(func_id, DATA_RAW_DIR)
    dim = X_updated.shape[1]

    # Dimension-aware exploration (simple and explainable)
    if dim >= 6:
        kappa = 2.5
    elif dim >= 4:
        kappa = 2.0
    else:
        kappa = 1.8

    # Fit GP on standardized y
    gp, y_mean, y_std = fit_gp_with_y_standardization(X_updated, y_updated, random_state=42)

    # Candidate points
    X_candidates = generate_random_candidates(n_candidates=N_CANDIDATES, dim=dim)

    # Predict on standardized scale
    mean_stdzd, std_stdzd = gp.predict(X_candidates, return_std=True)

    # UCB score on standardized scale (valid because it’s consistent scale)
    scores = acquisition_ucb(mean_stdzd, std_stdzd, kappa=kappa)

    # Select best candidate
    x_next = X_candidates[int(scores.argmax())]
    submission = format_submission(x_next)

    ok, msg = validate_submission_string(submission, dim=dim)
    if not ok:
        raise ValueError(f"Function {func_id}: invalid submission: {msg} -> {submission}")

    results_week3[func_id] = submission
    print(f"Week 3 - Function {func_id} (dim={dim}, kappa={kappa}): {submission}")


c:\Users\yshah\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\yshah\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Week 3 - Function 1 (dim=2, kappa=1.8): 0.773956-0.438878
Week 3 - Function 2 (dim=2, kappa=1.8): 0.773956-0.438878
Week 3 - Function 3 (dim=3, kappa=1.8): 0.708862-0.999390-0.007483
Week 3 - Function 4 (dim=4, kappa=2.0): 0.381611-0.390023-0.489693-0.455284
Week 3 - Function 5 (dim=4, kappa=2.0): 0.963141-0.994441-0.801624-0.999286
Week 3 - Function 6 (dim=5, kappa=2.0): 0.268756-0.730815-0.162103-0.932093-0.000434
Week 3 - Function 7 (dim=6, kappa=2.5): 0.219388-0.134546-0.344610-0.186044-0.178171-0.647537
Week 3 - Function 8 (dim=8, kappa=2.5): 0.043177-0.514701-0.082182-0.986416-0.832375-0.826257-0.077916-0.007706


### 5.7 Save Week 3 submissions

In [21]:
out_dir = PROJECT_ROOT / "outputs" / "week_03"
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "submissions.txt"
with out_file.open("w", encoding="utf-8") as f:
    for func_id in range(1, 9):
        f.write(f"Function {func_id}: {results_week3[func_id]}\n")

print("Saved Week 3 submissions to:", out_file)

Saved Week 3 submissions to: ..\outputs\week_03\submissions.txt


## Section 6 — Week 4 (Module 15): Strategy-based generation (adaptive exploration/exploitation)

### 6.1 Store Week 3 inputs and outputs

In [17]:
# Week 3 submitted inputs (portal)
WEEK3_INPUTS = {
    1: [0.773956, 0.438878],
    2: [0.773956, 0.438878],
    3: [0.708862, 0.999390, 0.007483],
    4: [0.381611, 0.390023, 0.489693, 0.455284],
    5: [0.963141, 0.994441, 0.801624, 0.999286],
    6: [0.268756, 0.730815, 0.162103, 0.932093, 0.000434],
    7: [0.219388, 0.134546, 0.344610, 0.186044, 0.178171, 0.647537],
    8: [0.043177, 0.514701, 0.082182, 0.986416, 0.832375, 0.826257, 0.077916, 0.007706],
}

# Week 3 observed outputs (email)
WEEK3_OUTPUTS = {
    1: 9.656222973626152e-41,
    2: 0.19196354967373935,
    3: -0.1388977457468655,
    4: -2.0470834341233686,
    5: 5394.580680183886,
    6: -1.281015418338123,
    7: 2.23531139550533,
    8: 8.9822261407959,
}


### 6.2 Helper: load training data up to Week 3 (initial + Week1 + Week2 + Week3)

In [18]:
def load_xy_with_week1_week2_week3(func_id: int, data_raw_dir):
    """
    Build the training dataset for Week 4 generation:
    initial + Week 1 + Week 2 + Week 3
    """
    X0, y0 = load_xy(func_id, data_raw_dir)
    dim = X0.shape[1]

    x_w1 = np.array(WEEK1_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_w1 = np.array([WEEK1_OUTPUTS[func_id]], dtype=float)

    x_w2 = np.array(WEEK2_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_w2 = np.array([WEEK2_OUTPUTS[func_id]], dtype=float)

    x_w3 = np.array(WEEK3_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_w3 = np.array([WEEK3_OUTPUTS[func_id]], dtype=float)

    if x_w1.shape[1] != dim or x_w2.shape[1] != dim or x_w3.shape[1] != dim:
        raise ValueError(f"Function {func_id}: dimension mismatch in weekly inputs.")

    X_updated = np.vstack([X0, x_w1, x_w2, x_w3])
    y_updated = np.concatenate([y0, y_w1, y_w2, y_w3])

    return X_updated, y_updated


### 6.3 Sanity check (points increased by 3)

In [19]:
for func_id in range(1, 9):
    X0, y0 = load_xy(func_id, DATA_RAW_DIR)
    X_updated, y_updated = load_xy_with_week1_week2_week3(func_id, DATA_RAW_DIR)
    print(
        f"Function {func_id}: initial={len(y0)} -> now={len(y_updated)} | dim={X_updated.shape[1]}"
    )


Function 1: initial=10 -> now=13 | dim=2
Function 2: initial=10 -> now=13 | dim=2
Function 3: initial=15 -> now=18 | dim=3
Function 4: initial=30 -> now=33 | dim=4
Function 5: initial=20 -> now=23 | dim=4
Function 6: initial=20 -> now=23 | dim=5
Function 7: initial=30 -> now=33 | dim=6
Function 8: initial=40 -> now=43 | dim=8


### 6.4 Strategy map for Week 4 (adaptive per function)

In [20]:
# Group A: boost exploration (poor / unstable results)
EXPLORATION_FUNCS = {1, 3, 4, 6}

# Group B: balanced (some signal, but not stable)
BALANCED_FUNCS = {2, 7, 8}

# Group C: careful exploitation (strong signal)
EXPLOIT_FUNCS = {5}

def get_week4_params(func_id: int, dim: int):
    """
    Returns (kappa, n_candidates) for Week 4 based on our analysis.
    """
    if func_id in EXPLORATION_FUNCS:
        return 3.0, 10000
    if func_id in BALANCED_FUNCS:
        # Slightly higher exploration for very high dimension
        if dim >= 8:
            return 2.5, 10000
        return 2.0, 8000
    if func_id in EXPLOIT_FUNCS:
        return 1.3, 6000

    # Fallback (shouldn't happen)
    return 2.0, 8000


### .5 Fit GP with y standardisation (FULL code, self-contained)

In [21]:
import numpy as np

def fit_gp_with_y_standardization(
    X: np.ndarray,
    y: np.ndarray,
    *,
    length_scale: float = 0.2,
    kernel_amplitude: float = 1.0,
    random_state: int = 42,
):
    """
    Standardize y -> fit GP -> return (gp, y_mean, y_std).

    Why:
      - outputs can be on very different scales
      - standardizing makes GP training and uncertainty more stable

    Notes:
      - GP is trained on standardized y (mean ~0, std ~1)
      - If you ever want predicted y in original scale:
            mean_real = mean_std * y_std + y_mean
            std_real  = std_std * y_std
    """
    y = np.asarray(y).reshape(-1)

    y_mean = float(np.mean(y))
    y_std_raw = float(np.std(y))

    # Avoid divide-by-zero if y is constant
    y_std = y_std_raw if y_std_raw > 0 else 1.0

    y_stdzd = (y - y_mean) / y_std

    gp = fit_gp(
        X,
        y_stdzd,
        length_scale=length_scale,
        kernel_amplitude=kernel_amplitude,
        random_state=random_state,
    )

    return gp, y_mean, y_std


In [22]:
def generate_random_candidates(
    n_candidates: int,
    dim: int,
    seed: int = 42,
) -> np.ndarray:
    """
    Generate random candidate points uniformly in [0, 1)^dim.
    """
    rng = np.random.default_rng(seed)
    Xc = rng.uniform(0.0, 1.0, size=(n_candidates, dim))

    # Make sure values are strictly < 1.0 (portal constraint)
    Xc = np.minimum(Xc, np.nextafter(1.0, 0.0))

    return Xc


#### Quick “do we have all required functions?” check

In [23]:
required_names = [
    "load_xy",
    "fit_gp",
    "acquisition_ucb",
    "format_submission",
    "validate_submission_string",
    "load_xy_with_week1_week2_week3",
    "fit_gp_with_y_standardization",
    "generate_random_candidates",
]

missing = [name for name in required_names if name not in globals()]
if missing:
    raise NameError(f"Missing required functions in notebook scope: {missing}")
else:
    print("All required functions are available ✅")


All required functions are available ✅


### 6.6 Generate Week 4 query points (one per function)

In [24]:
results_week4 = {}

for func_id in range(1, 9):
    X_train, y_train = load_xy_with_week1_week2_week3(func_id, DATA_RAW_DIR)
    dim = X_train.shape[1]

    kappa, n_candidates = get_week4_params(func_id, dim)

    # Fit GP on standardized y
    gp, y_mean, y_std = fit_gp_with_y_standardization(X_train, y_train, random_state=42)

    # Generate candidates
    X_candidates = generate_random_candidates(n_candidates=n_candidates, dim=dim)

    # Predict (standardized space)
    mean_stdzd, std_stdzd = gp.predict(X_candidates, return_std=True)

    # UCB scoring
    scores = acquisition_ucb(mean_stdzd, std_stdzd, kappa=kappa)

    # Best next x
    x_next = X_candidates[int(np.argmax(scores))]
    submission = format_submission(x_next)

    # Validate
    ok, msg = validate_submission_string(submission, dim=dim)
    if not ok:
        raise ValueError(f"Function {func_id}: invalid submission: {msg} -> {submission}")

    results_week4[func_id] = submission
    print(
        f"Week 4 - Function {func_id}: {submission} "
        f"(dim={dim}, kappa={kappa}, candidates={n_candidates})"
    )


c:\Users\yshah\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\yshah\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Week 4 - Function 1: 0.858598-0.697368 (dim=2, kappa=3.0, candidates=10000)
Week 4 - Function 2: 0.858598-0.697368 (dim=2, kappa=2.0, candidates=8000)
Week 4 - Function 3: 0.557032-0.783898-0.664314 (dim=3, kappa=3.0, candidates=10000)
Week 4 - Function 4: 0.008824-0.242528-0.978455-0.989068 (dim=4, kappa=3.0, candidates=10000)
Week 4 - Function 5: 0.901642-0.984569-0.982367-0.952999 (dim=4, kappa=1.3, candidates=6000)
Week 4 - Function 6: 0.232303-0.082896-0.719888-0.502509-0.047595 (dim=5, kappa=3.0, candidates=10000)
Week 4 - Function 7: 0.156095-0.329410-0.357365-0.206310-0.075234-0.768486 (dim=6, kappa=2.0, candidates=8000)
Week 4 - Function 8: 0.006934-0.051230-0.592215-0.084305-0.868502-0.759444-0.008818-0.573245 (dim=8, kappa=2.5, candidates=10000)


### 6.7 Save Week 4 submissions

In [25]:
out_dir = PROJECT_ROOT / "outputs" / "week_04"
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "submissions.txt"
with out_file.open("w", encoding="utf-8") as f:
    for func_id in range(1, 9):
        f.write(f"Function {func_id}: {results_week4[func_id]}\n")

print("Saved Week 4 submissions to:", out_file)


Saved Week 4 submissions to: ..\outputs\week_04\submissions.txt
